In [46]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,Dense

In [47]:
max_features=10000 
(X_train,y_train),(X_test,y_test) = imdb.load_data(num_words=max_features)

print(f"{X_train.shape},{y_train.shape}")
print(f"{X_test.shape},{y_test.shape}")


(25000,),(25000,)
(25000,),(25000,)


In [48]:
max_len = 500
X_train = sequence.pad_sequences(X_train,maxlen=max_len)
X_test = sequence.pad_sequences(X_test,maxlen=max_len)
X_train

array([[   0,    0,    0, ...,   19,  178,   32],
       [   0,    0,    0, ...,   16,  145,   95],
       [   0,    0,    0, ...,    7,  129,  113],
       ...,
       [   0,    0,    0, ...,    4, 3586,    2],
       [   0,    0,    0, ...,   12,    9,   23],
       [   0,    0,    0, ...,  204,  131,    9]], dtype=int32)

#### Pre padding done in here


# TRAIN SIMPLE RNN

In [49]:
max_features=10000
dim=128
neurons=128

In [50]:
model=Sequential()

model.add(Embedding(max_features,dim,input_length=max_len)) ## Embedding layer

model.add(SimpleRNN(neurons,activation='relu'))

model.add(Dense(1,activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [51]:
model.build(input_shape=(None,max_len))

In [52]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

In [53]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [54]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [55]:
history = model.fit(
    X_train,y_train,epochs=20,batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping]
)

Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 28s 42ms/step - accuracy: 0.6247 - loss: 14427.5420 - val_accuracy: 0.6264 - val_loss: 0.6238
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 40ms/step - accuracy: 0.7244 - loss: 87.2935 - val_accuracy: 0.6992 - val_loss: 0.5535
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 40ms/step - accuracy: 0.8289 - loss: 0.3859 - val_accuracy: 0.7848 - val_loss: 0.4643
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 40ms/step - accuracy: 0.8645 - loss: 0.3209 - val_accuracy: 0.7740 - val_loss: 0.4821
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.9038 - loss: 0.2473 - val_accuracy: 0.8452 - val_loss: 0.4244
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.9293 - loss: 0.1936 - val_accuracy: 0.8180 - val_loss: 0.4581
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 40ms/step - accuracy: 0.9466 - loss: 0.1483 - val_accuracy: 0.8332 - val_loss: 0.4761
Epoch 8/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 40ms/step - accuracy: 0.9643 - loss: 0.10

In [57]:
model.save('artifacts/simple_rnn.h5')